In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# STEP 1: LOAD AND PREPARE THE DATA
# ==========================================

# Read the CSV file
# The file has columns: v1 (spam/ham), v2 (message text)
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep only the first two columns (v1 and v2)
df = df[['v1', 'v2']]

# Rename columns for clarity
df.columns = ['label', 'message']

# Convert labels: 'spam' -> 1, 'ham' -> 0
df['label'] = df['label'].map({'spam': 1, 'ham': 0})

# Remove any rows with missing values
df = df.dropna()

# Show first few rows to verify
print("First 5 rows of dataset:")
print(df.head())
print("\n" + "="*50 + "\n")

# ==========================================
# STEP 2: SPLIT DATA INTO TRAIN AND TEST SETS
# ==========================================

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    df['message'],      # The text messages
    df['label'],        # The labels (spam=1, ham=0)
    test_size=0.2,      # 20% for testing
    random_state=42     # For reproducible results
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print("\n" + "="*50 + "\n")

# ==========================================
# STEP 3: CONVERT TEXT TO NUMBERS USING TF-IDF
# ==========================================

# Create TF-IDF Vectorizer
# - max_features=5000: Use top 5000 words
# - stop_words='english': Remove common words like 'the', 'and', etc.
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# Convert training messages into TF-IDF features
X_train_tfidf = tfidf.fit_transform(X_train)

# Convert testing messages using the same vectorizer
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF features shape (training): {X_train_tfidf.shape}")
print("Each message is now represented as 5000 numbers (word importance scores)")
print("\n" + "="*50 + "\n")

# ==========================================
# STEP 4: TRAIN THE CLASSIFIER
# ==========================================

# Use Naive Bayes classifier (good for text classification)
model = MultinomialNB()

# Train the model
model.fit(X_train_tfidf, y_train)

print("Model training complete!")
print("\n" + "="*50 + "\n")

# ==========================================
# STEP 5: MAKE PREDICTIONS AND EVALUATE
# ==========================================

# Predict on test data
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Show detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))

# Show confusion matrix
print("\nConfusion Matrix:")
print("(Rows: Actual, Columns: Predicted)")
print("[ [True Ham, False Spam]")
print("  [False Ham, True Spam] ]")
print(confusion_matrix(y_test, y_pred))

print("\n" + "="*50 + "\n")

# ==========================================
# STEP 6: TEST WITH NEW MESSAGES
# ==========================================

def predict_message(message):
    """
    Function to predict if a message is spam or ham
    """
    # Convert the message to TF-IDF features
    message_tfidf = tfidf.transform([message])
    
    # Predict
    prediction = model.predict(message_tfidf)[0]
    
    # Return result
    if prediction == 1:
        return "SPAM"
    else:
        return "HAM"

# Test with some example messages
print("Testing with new messages:")
print("-" * 40)

test_messages = [
    "Congratulations! You've won a free iPhone! Call now!",
    "Hey, want to meet for coffee tomorrow?",
    "URGENT: Your account has been compromised. Click here to verify.",
    "I'll be home at 6pm, don't forget to buy milk.",
    "FREE entry in 2 a wkly comp to win FA Cup final tickets!"
]

for msg in test_messages:
    result = predict_message(msg)
    print(f"Message: {msg[:50]}...")
    print(f"Prediction: {result}")
    print("-" * 40)

# ==========================================
# EXTRA: SEE MOST IMPORTANT WORDS FOR SPAM
# ==========================================

print("\n" + "="*50)
print("Most important words for detecting SPAM:")
print("-" * 40)

# Get feature names (words)
feature_names = tfidf.get_feature_names_out()

# Get the coefficients (importance) from the model
# Higher coefficient means more indicative of SPAM
spam_coefficients = model.feature_log_prob_[1]  # Class 1 = Spam
ham_coefficients = model.feature_log_prob_[0]   # Class 0 = Ham

# Calculate difference (spam importance - ham importance)
importance = spam_coefficients - ham_coefficients

# Get top 10 words that indicate SPAM
top_spam_indices = importance.argsort()[-10:][::-1]

print("Top 10 words that suggest a message is SPAM:")
for idx in top_spam_indices:
    print(f"  - {feature_names[idx]}")

print("\n" + "="*50)
print("Model ready! You can now detect spam messages.")

First 5 rows of dataset:
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


Training samples: 4457
Testing samples: 1115


TF-IDF features shape (training): (4457, 5000)
Each message is now represented as 5000 numbers (word importance scores)


Model training complete!


Model Accuracy: 97.31%

Classification Report:
              precision    recall  f1-score   support

     Ham (0)       0.97      1.00      0.98       965
    Spam (1)       1.00      0.80      0.89       150

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
(Rows: Actual, Columns: 